# Zero-Shot Classification — Google AI Studio (Gemini)

Testa modelos Gemini sem qualquer treino nos dados.  
O modelo recebe o texto e tem de escolher a label correcta entre as 5 classes.

**Classes:** `Human · Google · Meta · OpenAI · Anthropic`

**Pipeline:**
```
test.csv → prompt formatado → Gemini API → parse label → accuracy + report
```

## 1. Instalar dependências

In [1]:
# !pip install google-genai python-dotenv pandas scikit-learn tqdm

In [2]:
import os
import time
import pandas as pd
from google import genai
from dotenv import load_dotenv
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

load_dotenv()
print("Imports OK")

Imports OK


## 2. Configuração

In [ ]:
# ── API Key ───────────────────────────────────────────────────────────────────
# Obtém em: https://aistudio.google.com/app/apikey
# Guarda num ficheiro .env na mesma pasta:  API_KEY=a_tua_key
API_KEY = "key"
client  = genai.Client(api_key=API_KEY)

# ── Modelo ────────────────────────────────────────────────────────────────────
# Opções disponíveis no AI Studio (descomenta o que quiseres testar):
#MODEL_NAME = "gemini-2.0-flash"      # rápido e barato — bom para começar
# MODEL_NAME = "gemini-2.0-flash-lite"
# MODEL_NAME = "gemini-2.5-pro-preview-03-25"   # mais capaz, mais lento
MODEL_NAME = "gemma-3-27b-it"                 # Gemma open-weight via AI Studio

# ── Dataset ───────────────────────────────────────────────────────────────────
TEST_PATH  = "data/test.csv"
TEXT_COL   = "Text"
LABEL_COL  = "Label"
LABELS     = ["Human", "Google", "Meta", "OpenAI", "Anthropic"]

# Quantos exemplos testar (None = todos)
# Começa com 100-200 para estimar tempo antes de correr tudo
SUBSET     = None

# Pausa entre pedidos (segundos) — evita rate limit
# Flash: 15 RPM no tier gratuito → mínimo 4s entre pedidos
SLEEP_SEC  = 2.0

SEED = 42
print(f"Modelo  : {MODEL_NAME}")
print(f"Dataset : {TEST_PATH}")
print(f"Subset  : {SUBSET if SUBSET else 'todos'}")

Modelo  : gemma-3-27b-it
Dataset : data/test.csv
Subset  : todos


## 3. Carregar Dataset

In [4]:
df = pd.read_csv(TEST_PATH)

if SUBSET:
    df = df.sample(SUBSET, random_state=SEED).reset_index(drop=True)

print(f"Exemplos a classificar: {len(df)}")
print(f"\nDistribuição de classes:")
print(df[LABEL_COL].value_counts())

FileNotFoundError: [Errno 2] No such file or directory: 'data/test.csv'

## 4. Prompt

O prompt instrui o modelo a devolver **apenas** a label — sem explicações.  
Isto simplifica o parsing e reduz tokens de output.

In [ ]:
SYSTEM_PROMPT = """You are an expert at identifying whether a text was written by a human or generated by a specific AI system.
Your task is to classify the source of the given text.

The possible sources are:
- Human: written by a person
- Google: generated by a Google AI model (e.g. Gemma, Gemini)
- Meta: generated by a Meta AI model (e.g. LLaMA)
- OpenAI: generated by an OpenAI model (e.g. GPT-3, GPT-4, ChatGPT)
- Anthropic: generated by an Anthropic model (e.g. Claude)

Rules:
1. Reply with ONLY one of these exact words: Human, Google, Meta, OpenAI, Anthropic
2. Do not include any explanation, punctuation, or extra text
3. Your entire response must be a single word from the list above"""


def build_prompt(text: str) -> str:
    return f"""Classify the source of the following text.

Text:
{text.strip()}

Source (reply with one word only):"""


# Pré-visualizar o prompt com um exemplo
sample_text = df[TEXT_COL].iloc[0]
sample_label = df[LABEL_COL].iloc[0]

print("=== SYSTEM PROMPT ===")
print(SYSTEM_PROMPT)
print("\n=== USER PROMPT (exemplo) ===")
print(build_prompt(sample_text[:300] + "..."))
print(f"\n[Label real: {sample_label}]")

## 5. Inicializar Modelo Gemini

In [ ]:
# Teste rápido de conectividade
response = client.models.generate_content(
    model=MODEL_NAME,
    contents="Hello, how are you?",
    config=genai.types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        temperature=0.0,
        max_output_tokens=10,
    )
)
print(f"Teste de ligação OK. Resposta: '{response.text.strip()}'")

## 6. Função de Classificação e Parsing

In [ ]:
def parse_label(raw: str) -> str:
    """
    Normaliza a resposta do modelo para uma das 5 labels válidas.
    Retorna 'INVALID' se não conseguir fazer match.
    """
    cleaned = raw.strip().strip('.,!?"\' ').capitalize()

    # Match exacto
    if cleaned in LABELS:
        return cleaned

    # Match case-insensitive
    cleaned_lower = cleaned.lower()
    for label in LABELS:
        if label.lower() == cleaned_lower:
            return label

    # Match parcial (ex: "OpenAI model" → "OpenAI")
    for label in LABELS:
        if label.lower() in cleaned_lower:
            return label

    return "INVALID"


def classify_text(text: str, retries: int = 3) -> str:
    """Envia um texto ao Gemini e devolve a label prevista."""
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=build_prompt(text),
                config=genai.types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    temperature=0.0,
                    max_output_tokens=10,
                )
            )
            return parse_label(response.text)
        except Exception as e:
            wait = SLEEP_SEC * (attempt + 2)
            print(f"  [Erro tentativa {attempt+1}] {e} — a aguardar {wait:.0f}s...")
            time.sleep(wait)
    return "INVALID"


# Testar parsing
assert parse_label("human")    == "Human"
assert parse_label(" OpenAI ") == "OpenAI"
assert parse_label("ANTHROPIC") == "Anthropic"
assert parse_label("Meta AI")  == "Meta"
print("parse_label OK")

## 7. Classificação do Dataset

> ⚠️ Esta célula faz N chamadas à API. Com `SLEEP_SEC=4` e `SUBSET=200`, demora ~15 minutos.  
> Os resultados são guardados em `results_df` para não perderes progresso.

In [ ]:
predictions = []
raw_responses = []
invalid_count = 0

for i, row in tqdm(df.iterrows(), total=len(df), desc=f"{MODEL_NAME}"):
    pred = classify_text(str(row[TEXT_COL]))
    predictions.append(pred)

    if pred == "INVALID":
        invalid_count += 1
        print(f"  [INVALID] idx={i} | label real={row[LABEL_COL]}")

    time.sleep(SLEEP_SEC)

# Guardar resultados
results_df = df.copy()
results_df["predicted"] = predictions

print(f"\nClassificação concluída.")
print(f"  Respostas inválidas : {invalid_count}/{len(df)} ({invalid_count/len(df)*100:.1f}%)")

# Guardar CSV para não perder resultados
output_path = f"results_zeroshot_{MODEL_NAME.replace('/', '_')}.csv"
results_df.to_csv(output_path, index=False)
print(f"  Resultados guardados em: {output_path}")

## 8. Métricas e Resultados

In [ ]:
# Filtrar respostas inválidas para não distorcer as métricas
valid_df = results_df[results_df["predicted"] != "INVALID"].copy()

if invalid_count > 0:
    print(f"Nota: {invalid_count} respostas inválidas excluídas das métricas.")
    print(f"Exemplos avaliados: {len(valid_df)}/{len(results_df)}\n")

true_labels = valid_df[LABEL_COL].tolist()
pred_labels = valid_df["predicted"].tolist()

acc = accuracy_score(true_labels, pred_labels)
print(f"{'='*50}")
print(f"Modelo         : {MODEL_NAME}")
print(f"Exemplos       : {len(valid_df)}")
print(f"Zero-Shot Acc  : {acc:.4f}  ({acc*100:.1f}%)")
print(f"{'='*50}\n")
print(classification_report(true_labels, pred_labels,
                             labels=LABELS, zero_division=0))

In [ ]:
# Matriz de confusão
cm = confusion_matrix(true_labels, pred_labels, labels=LABELS)
ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(cmap="Blues", colorbar=False)
plt.title(f"Zero-Shot — {MODEL_NAME}  (acc={acc:.3f})")
plt.tight_layout()
plt.show()

In [ ]:
# Distribuição das previsões vs labels reais
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

results_df[LABEL_COL].value_counts().reindex(LABELS, fill_value=0).plot(
    kind="bar", ax=axes[0], color="steelblue", edgecolor="white"
)
axes[0].set_title("Distribuição — Labels Reais")
axes[0].set_xlabel(""); axes[0].set_ylabel("Contagem")
axes[0].tick_params(axis="x", rotation=30)

results_df["predicted"].value_counts().reindex(LABELS + ["INVALID"], fill_value=0).plot(
    kind="bar", ax=axes[1], color="darkorange", edgecolor="white"
)
axes[1].set_title("Distribuição — Previsões Gemini")
axes[1].set_xlabel(""); axes[1].set_ylabel("Contagem")
axes[1].tick_params(axis="x", rotation=30)

plt.suptitle(f"Zero-Shot {MODEL_NAME}", fontsize=13)
plt.tight_layout()
plt.show()

## 9. Erros mais frequentes

In [ ]:
# Ver exemplos onde o modelo errou
errors_df = valid_df[valid_df[LABEL_COL] != valid_df["predicted"]].copy()
print(f"Total de erros: {len(errors_df)} / {len(valid_df)} ({len(errors_df)/len(valid_df)*100:.1f}%)\n")

print("Confusões mais frequentes (real → previsto):")
print(
    errors_df.groupby([LABEL_COL, "predicted"])
    .size()
    .sort_values(ascending=False)
    .head(10)
    .to_string()
)

In [ ]:
# Ver os primeiros N exemplos errados com o texto
N_SHOW = 5
print(f"Primeiros {N_SHOW} erros:\n")
for _, row in errors_df.head(N_SHOW).iterrows():
    print(f"Real     : {row[LABEL_COL]}")
    print(f"Previsto : {row['predicted']}")
    print(f"Texto    : {str(row[TEXT_COL])[:200]}...")
    print("-" * 60)

## 10. Comparar múltiplos modelos (opcional)

Se correste a célula 7 com modelos diferentes e guardaste os CSVs, usa esta célula para comparar.

In [ ]:
# Lista de ficheiros de resultados gerados pela célula 7
result_files = [
    # ("gemini-1.5-flash",  "results_zeroshot_gemini-1.5-flash.csv"),
    # ("gemini-1.5-pro",    "results_zeroshot_gemini-1.5-pro.csv"),
    # ("gemini-2.0-flash",  "results_zeroshot_gemini-2.0-flash.csv"),
]

if result_files:
    rows = []
    for model_name, fpath in result_files:
        rdf  = pd.read_csv(fpath)
        vdf  = rdf[rdf["predicted"] != "INVALID"]
        acc  = accuracy_score(vdf[LABEL_COL], vdf["predicted"])
        rows.append({"Model": model_name, "Accuracy": round(acc, 4),
                     "Valid": len(vdf), "Invalid": len(rdf) - len(vdf)})

    summary = pd.DataFrame(rows).sort_values("Accuracy", ascending=False)
    print(summary.to_string(index=False))

    summary.plot(kind="bar", x="Model", y="Accuracy",
                 legend=False, color="steelblue", edgecolor="white", figsize=(8, 4))
    plt.title("Zero-Shot Accuracy por Modelo")
    plt.ylabel("Accuracy"); plt.xticks(rotation=15)
    plt.ylim(0, 1); plt.tight_layout(); plt.show()
else:
    print("Descomenta os ficheiros de resultados para comparar modelos.")